# 主动寻找最简单的作用方向

矩阵 A=[[3,1],[1,3]] 把两个坐标混在一起。我们先找哪些输入只乘一个数，再用这些方向分解普通输入，最后连接零空间。

依赖 NumPy、Matplotlib。按新内核顺序运行；图中为英文，避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":11})
A = np.array([[3.,1.],[1.,3.]])
v1 = np.array([1.,1.])
v2 = np.array([1.,-1.])


## 1. 普通输入为什么不能直接说乘几倍？

先算 A(1,0)，再作用一次。对 (3,1)，两个坐标下一次各乘了几倍？这两个倍数是否相同？

In [ ]:
ordinary = np.array([1.,0.])
once = A @ ordinary
twice = A @ once
print("input:",ordinary,"one step:",once,"two steps:",twice)
print("coordinate ratios from (3,1):",twice/once)
assert np.allclose(once,[3,1])
assert np.allclose(twice,[10,6])
assert not np.isclose((twice/once)[0],(twice/once)[1])


## 2. 先试相等，再试相反

不看输出，先算 A(1,1) 与 A(1,-1)。

然后同时缩放两个输入坐标，试 (2,2)、(-1,-1)、(3,-3)。只乘一个数的规则是否依然成立？

In [ ]:
for vector in [v1,2*v1,-v1,v2,3*v2]:
    output = A @ vector
    print("input:",vector,"output:",output,"ratios:",output/vector)
assert np.allclose(A @ v1,4*v1)
assert np.allclose(A @ v2,2*v2)


## 3. 看所在的直线

灰色箭头为输入，绿色虚线为输出。后两幅图中的两根箭头在同一条直线上。

只有两幅成功图，还不能说明整个平面都可这样计算；还要检查这些方向能不能拆开普通输入。

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(12,4),dpi=150)
for ax,vector,title in zip(axes,[ordinary,v1,v2],
                          ["Ordinary input","Equal coordinates","Opposite coordinates"]):
    output = A @ vector
    ax.annotate("",xy=output,xytext=(0,0),
                arrowprops={"arrowstyle":"->","color":"#23836b","lw":2.5,"linestyle":"--"})
    ax.annotate("",xy=vector,xytext=(0,0),
                arrowprops={"arrowstyle":"->","color":"#64717a","lw":2.5})
    ax.scatter(*vector,color="#64717a",s=25,label="Input")
    ax.scatter(*output,color="#23836b",s=25,label="Output")
    low = np.minimum.reduce([vector,output,np.zeros(2)])-0.6
    high = np.maximum.reduce([vector,output,np.zeros(2)])+0.8
    ax.set(xlim=(low[0],high[0]),ylim=(low[1],high[1]),
           aspect="equal",title=title,xlabel="x1",ylabel="x2")
    ax.axhline(0,color="0.7",lw=0.8)
    ax.axvline(0,color="0.7",lw=0.8)
    ax.grid(alpha=0.2)
axes[0].legend(loc="upper left")
fig.tight_layout()
plt.show()


## 4. 手算普通输入的两份贡献

把 (3,1) 写成 c1(1,1)+c2(1,-1)。先解 c1+c2=3、c1-c2=1。

得到 (2,1) 后，预测矩阵作用后的两个系数，而不是先做原矩阵乘法。再修改 x，检查规则。

In [ ]:
x = np.array([3.,1.])  # Also try [5.,1.] or [0.,2.].
c1 = (x[0]+x[1])/2
c2 = (x[0]-x[1])/2
decomposition = c1*v1+c2*v2
separate_output = (4*c1)*v1+(2*c2)*v2
print("input:",x,"coefficients:",[c1,c2])
print("after scaling coefficients:",[4*c1,2*c2])
print("separate output:",separate_output,"matrix output:",A @ x)
assert np.allclose(decomposition,x)
assert np.allclose(separate_output,A @ x)


## 5. 把向量相加画出来

蓝箭头是第一份贡献；橙箭头从它的末端出发，表示加上第二份贡献。虚线是总向量。

先看作用前的两个系数，再看作用后的系数。两图各自自动选择刻度，不用像素长度比较倍数。

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(10,4.5),dpi=150)
for ax,a,b,title in [(axes[0],c1,c2,"Before: independent contributions"),
                     (axes[1],4*c1,2*c2,"After: scale the coefficients")]:
    first,second = a*v1,b*v2
    result = first+second
    for start,end,color in [(np.zeros(2),first,"#2477ad"),
                             (first,result,"#d36a25")]:
        ax.annotate("",xy=end,xytext=start,
                    arrowprops={"arrowstyle":"->","color":color,"lw":2.5})
    ax.annotate("",xy=result,xytext=(0,0),
                arrowprops={"arrowstyle":"->","color":"#23836b","lw":1.8,"linestyle":"--"})
    points = np.array([np.zeros(2),first,result])
    low,high = points.min(axis=0)-0.8,points.max(axis=0)+0.8
    ax.set(xlim=(low[0],high[0]),ylim=(low[1],high[1]),
           aspect="equal",xlabel="x1",ylabel="x2",title=title)
    ax.axhline(0,color="0.7",lw=0.8)
    ax.axvline(0,color="0.7",lw=0.8)
    ax.grid(alpha=0.2)
    ax.text(0.04,0.95,f"coefficients: ({a:g}, {b:g})",
            transform=ax.transAxes,va="top")
fig.tight_layout()
plt.show()


## 6. 重复作用不用重新混合

初始系数 c1、c2，经过k次后分别乘4^k、2^k。

先手算两次后的系数，再比较迭代与直接公式。没有先调用特征值求解器。

In [ ]:
state = x.copy()
for k in range(6):
    formula = c1*4**k*v1+c2*2**k*v2
    print("step",k,"coefficients:",[c1*4**k,c2*2**k],"output:",state)
    assert np.allclose(state,formula)
    state = A @ state


## 7. 找特征方向，回到零空间

由 Av=lambda v，得到 (A-lambda I)v=0。

先手算 lambda=4 时两行要求什么，再算 lambda=2。尝试 lambda=3：为什么只得到零向量？

In [ ]:
for value in [4.,2.,3.]:
    shifted = A-value*np.eye(2)
    print("lambda:",value,"matrix:\n",shifted,
          "rank:",np.linalg.matrix_rank(shifted))
assert np.allclose((A-4*np.eye(2)) @ v1,0)
assert np.allclose((A-2*np.eye(2)) @ v2,0)
assert np.linalg.matrix_rank(A-3*np.eye(2))==2


## 8. 改矩阵，预测哪些东西不变

把两个对角线上的3同时改成d，非对角线仍为1。先算相等与相反输入，预测两个倍数。

试 d=1：原来的方向有没有变？其中一个输出为什么成了零？再试 d=0，观察负倍数。

In [ ]:
d = 1.  # Try 3, 0, -2.
changed = np.array([[d,1.],[1.,d]])
print("equal-coordinate output:",changed @ v1,"multiplier:",d+1)
print("opposite-coordinate output:",changed @ v2,"multiplier:",d-1)
assert np.allclose(changed @ v1,(d+1)*v1)
assert np.allclose(changed @ v2,(d-1)*v2)


## 9. 实矩阵一定有实特征方向吗？

把规则换成旋转90度。先想：哪个非零实向量旋转后仍在原来的直线上？

下面用两列行列式检查原向量与输出是否在同一直线上。对输入(a,b)，这个数是 a^2+b^2，非零输入时永远不是零。

In [ ]:
Q = np.array([[0.,-1.],[1.,0.]])
for vector in [np.array([1.,0.]),v1,v2,np.array([3.,1.])]:
    output = Q @ vector
    parallel_test = vector[0]*output[1]-vector[1]*output[0]
    print("input:",vector,"rotated:",output,"parallel test:",parallel_test)
    assert np.isclose(parallel_test,vector @ vector)
    assert parallel_test > 0


## 10. 应用：迁移也能拆成各自缩放

客户迁移矩阵 M=[[0.8,0.3],[0.2,0.7]]。先核对 (3,2) 的倍数为1，(1,-1) 的倍数为0.5。

把初始 (80,20) 写成这两个方向的贡献。为什么长期会靠近 (60,40)，而不是像前面的矩阵一样放大？

In [ ]:
M = np.array([[0.8,0.3],[0.2,0.7]])
fixed_direction = np.array([3.,2.])
difference_direction = np.array([1.,-1.])
assert np.allclose(M @ fixed_direction,fixed_direction)
assert np.allclose(M @ difference_direction,0.5*difference_direction)
initial = np.array([80.,20.])
assert np.allclose(initial,20*fixed_direction+20*difference_direction)
state = initial.copy()
for k in range(6):
    formula = 20*fixed_direction+20*0.5**k*difference_direction
    assert np.allclose(state,formula)
    print("day",k,"expected customers:",state)
    state = M @ state


## 自己重建

1. 找到一个特殊方向，为什么还不足以简化整个平面的输入？
2. 手算 (5,1) 在两条特征方向上的系数，再写出第k次作用。
3. d=1 时，哪个特征方向成为零空间方向？
4. 为什么不是每个实矩阵都能找到实特征基？

来源：[MIT 18.06 第二十一讲](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/e820a9d3c5709407ebbd6cf8a065d1ed_MIT18_06S10_L21.pdf)。